In [ ]:
!pip install pyopenms

     |████████████████████████████████| 40.0MB 103kB/s 


In [ ]:
from pyopenms import *
import numpy as np
import pandas as pd

In [ ]:
import csv
import sys

### Load data

In [ ]:
#class4
ec07 = pd.read_csv("EC07.tsv", sep='\t')
ec08 = pd.read_csv("EC08.tsv", sep='\t')
ec12 = pd.read_csv("EC12.tsv", sep='\t')
ec18 = pd.read_csv("EC18.tsv", sep='\t')
ec29 = pd.read_csv("EC29.tsv", sep='\t')

#class5
ec06 = pd.read_csv("EC06.tsv", sep='\t')
ec15 = pd.read_csv("EC15.tsv", sep='\t')
ec17 = pd.read_csv("EC17.tsv", sep='\t')
ec27 = pd.read_csv("EC27.tsv", sep='\t')
ec41 = pd.read_csv("EC41.tsv", sep='\t')

In [ ]:
#class4
ec07 = ec07.rename(columns={"b'label'": "label", "Intensity":"Intensity_07"})
ec08 = ec08.rename(columns={"b'label'": "label", "Intensity":"Intensity_08"})
ec12 = ec12.rename(columns={"b'label'": "label", "Intensity":"Intensity_12"})
ec18 = ec18.rename(columns={"b'label'": "label", "Intensity":"Intensity_18"})
ec29 = ec29.rename(columns={"b'label'": "label", "Intensity":"Intensity_29"})

#class5
ec06 = ec06.rename(columns={"b'label'": "label", "Intensity":"Intensity_06"})
ec15 = ec15.rename(columns={"b'label'": "label", "Intensity":"Intensity_15"})
ec17 = ec17.rename(columns={"b'label'": "label", "Intensity":"Intensity_17"})
ec27 = ec27.rename(columns={"b'label'": "label", "Intensity":"Intensity_27"})
ec41 = ec41.rename(columns={"b'label'": "label", "Intensity":"Intensity_41"})

In [ ]:
#class4
df_07 = ec07[['Intensity_07', 'label']]
df_08 = ec08[['Intensity_08', 'label']]
df_12 = ec12[['Intensity_12', 'label']]
df_18 = ec18[['Intensity_18', 'label']]
df_29 = ec29[['Intensity_29', 'label']]

#class5
df_06 = ec06[['Intensity_06', 'label']]
df_15 = ec15[['Intensity_15', 'label']]
df_17 = ec17[['Intensity_17', 'label']]
df_27 = ec27[['Intensity_27', 'label']]
df_41 = ec41[['Intensity_41', 'label']]

In [ ]:
print("class 4")
print("EC07 : ", len(df_07))
print("EC08 : ", len(df_08))
print("EC12 : ", len(df_12))
print("EC18 : ", len(df_18))
print("EC29 : ", len(df_29))
print()
print("class 5")
print("EC06 : ", len(df_06))
print("EC15 : ", len(df_15))
print("EC17 : ", len(df_17))
print("EC27 : ", len(df_27))
print("EC41 : ", len(df_41))

class 4
EC07 :  81890
EC08 :  77875
EC12 :  74643
EC18 :  72594
EC29 :  66593

class 5
EC06 :  81662
EC15 :  72370
EC17 :  72158
EC27 :  68098
EC41 :  38762


In [ ]:
df_4 = pd.merge(pd.merge(pd.merge(pd.merge(df_07, df_08, on='label', how='outer'), 
                                  df_12, on='label', how='outer'), 
                         df_18, on='label', how='outer'), 
                df_29, on='label', how='outer')
df_4 = df_4.set_index('label')
df_4.loc['class'] = 4
len(df_4)

216981

In [ ]:
df_5 = pd.merge(pd.merge(pd.merge(pd.merge(df_06, df_15, on='label', how='outer'), 
                                  df_17, on='label', how='outer'), 
                         df_27, on='label', how='outer'), 
                df_41, on='label', how='outer')
df_5 = df_5.set_index('label')
df_5.loc['class'] = 5
len(df_5)

217213

In [ ]:
all_df = pd.merge(df_4, df_5, on = 'label', how='outer')

In [ ]:
len(all_df)

280419

### drop features with 5 more nan values

In [ ]:
all_df['na_count'] = all_df.isnull().sum(axis=1)

In [ ]:
all_df.head(10)

In [ ]:
new_df = all_df[all_df.na_count < 5]
len(new_df)

10100

In [ ]:
df_final = new_df.drop(columns = ['na_count'])

### KNN imputation

In [ ]:
!pip install impyute

In [ ]:
import sys
from impyute.imputation.cs import fast_knn
sys.setrecursionlimit(100000) #Increase the recursion limit of the OS

# start the KNN training
impute_knn = fast_knn(df_final.values, k=30)

In [ ]:
print(np.count_nonzero(impute_knn < 0))

0


In [ ]:
len(imputed_training)

10100

In [ ]:
df = pd.DataFrame(impute_knn, index = df_final.index, columns = df_final.columns)

In [ ]:
df_fs = df.transpose()
df_fs.head()

label,158450,158461,159190,159897,158892,158910,158656,160007,158797,158803,159940,158881,160841,160814,160407,160487,160478,163391,163338,163368,161637,161662,161543,161439,159684,158885,158864,158867,160189,160176,159604,159581,159569,161980,162135,162127,160853,160843,159868,160559,...,195096,211460,226131,200016,241524,224587,222064,217342,213606,228213,216744,209228,233726,217211,225991,123074,214926,121203,229410,230283,227734,248991,226270,233907,226823,230306,230398,233420,232980,232983,134679,244057,248740,248216,248705,138699,251033,250764,123745,class
Intensity_07,1.240032e+07,1.231941e+07,1.154429e+07,7.641695e+06,7.005361e+06,6.909067e+06,6.650349e+06,6.480330e+06,6.347875e+06,6.339688e+06,6.338793e+06,6.121264e+06,5.670613e+06,5.658167e+06,5.644208e+06,5.642091e+06,5.640153e+06,5.555298e+06,5.176588e+06,5.173532e+06,5.130831e+06,5.129508e+06,5.117219e+06,4.688407e+06,4.602236e+06,4.596176e+06,4.592988e+06,4.592842e+06,4.589683e+06,4.589601e+06,4.539348e+06,4.537047e+06,4.521520e+06,4.458753e+06,4.455377e+06,4.447480e+06,4.374811e+06,4.370821e+06,4.322213e+06,4.320773e+06,...,201158.535692,318818.780946,317431.132535,318809.299514,262824.527007,317696.918958,301454.959791,310055.602155,320977.032176,311486.946074,317551.093675,320768.134222,191471.228380,317209.534328,309252.006004,287737.975904,318109.660364,268841.964461,306780.608199,305338.704925,251025.704896,309747.050451,318005.261575,194222.538037,310835.897100,304195.166532,306697.111772,163673.382393,140332.826578,140301.233768,2.239572e+05,245447.070002,258410.915437,287900.725815,282325.360409,193633.708051,212867.775827,177259.215651,296986.845922,4.0
Intensity_08,9.421432e+05,1.697851e+06,1.112890e+06,1.132575e+06,7.089609e+05,1.712300e+06,1.556633e+06,6.733256e+05,6.755060e+05,4.682238e+05,1.101520e+06,6.807162e+05,6.128656e+05,1.268163e+06,6.255141e+05,6.856406e+05,1.675242e+06,6.523443e+05,5.690666e+05,7.223731e+05,8.774286e+05,7.259568e+05,8.761257e+05,1.857612e+06,6.315183e+05,4.702782e+05,4.700014e+05,7.013632e+05,6.966578e+05,6.308739e+05,7.753972e+05,7.763030e+05,1.731920e+06,3.161361e+05,1.038503e+06,5.316747e+05,6.120876e+05,5.841974e+05,1.307534e+06,6.039778e+05,...,143791.885177,278153.939460,271939.291558,274581.012119,185111.522190,278153.939460,271861.970553,271913.581809,278153.939460,272177.181174,278153.939460,266350.362940,143506.938640,270980.386474,271402.745269,247077.181067,278153.939460,250454.656097,272680.755215,272067.914227,221819.486640,264326.081111,278153.939460,141877.202034,271818.956387,272409.514503,272475.557765,174070.567322,135272.361490,135682.587168,1.528802e+05,182191.406517,221632.164033,249165.733768,229382.750701,181920.194656,119579.554914,177799.402371,233470.407496,4.0
Intensity_12,1.209820e+06,6.615327e+05,6.022744e+05,7.282587e+05,6.387160e+05,1.025411e+06,6.278083e+05,6.004077e+05,6.036128e+05,5.855834e+05,6.138663e+05,7.550371e+05,5.617143e+05,4.987609e+05,5.512871e+05,1.082381e+06,5.212346e+05,6.247853e+05,7.871539e+05,7.019670e+05,4.800536e+05,7.111478e+05,6.075719e+05,6.769719e+05,1.030547e+06,5.784611e+05,1.024127e+06,6.061671e+05,1.020559e+06,5.901115e+05,7.210945e+05,5.870598e+05,5.943622e+05,5.540469e+05,3.455744e+05,5.426862e+05,6.055027e+05,2.867518e+05,6.496286e+05,5.303870e+05,...,196151.587512,400770.160228,400645.905298,395995.116496,385289.929782,400875.365844,400645.905298,400645.905298,400811.608842,400645.905298,400828.663374,393454.106260,203192.147409,400645.905298,400645.905298,318045.687041,400856.195684,330786.674699,400645.905298,400645.905298,398604.779015,400645.905298,400844.369894,240939.777894,400645.905298,400645.905298,400645.905298,272567.580547,283350.378595,284091.736992,2.091854e+05,388254.006941,243580.723471,398672.605235,397055.904393,305122.558552,233734.340985,216415.631137,372894.859151,4.0
Intensity_18,3.578857e+05,6.222612e+05,3.472172e+05,6.047481e+05,5.269836e+05,3.790239e+05,4.283461e+05,4.817727e+05,2.201880e+05,1.425997e+06,3.925517e+05,6.032640e+05,3.58

### Feature selection

In [ ]:
X = df_fs.drop(columns = ['class'])
Y = df_fs[['class']]

In [ ]:
bestfeatures = SelectKBest(score_func=chi2, k=100)
fit = bestfeatures.fit(X, Y)

dfscores = pd.DataFrame(fit.scores_)
dfcolumns = pd.DataFrame(X.columns)

#concat two dataframes for better visualization 
featureScores = pd.concat([dfcolumns, dfscores],axis=1)
featureScores.columns = ['Specs','Score']  #naming the dataframe columns

print(featureScores.nlargest(10, 'Score')) 

       Specs         Score
8883  251986  3.974351e+07
8998  259466  3.251096e+07
8891  259456  2.891784e+07
8381  261434  2.878700e+07
4364  261433  2.833461e+07
5248  250530  2.737571e+07
6487  252807  2.630472e+07
5538  252786  2.621682e+07
9352  262034  2.388487e+07
3899  233373  2.332088e+07


In [ ]:
top_100_features = featureScores.nlargest(100, 'Score')

In [ ]:
top_100_features['Specs'].values

array([251986, 259466, 259456, 261434, 261433, 250530, 252807, 252786,
       262034, 233373, 259438, 234307, 280385, 266504, 232941, 232933,
       232936, 259477, 266512, 266492, 251858, 251893, 251894, 250434,
       250331, 252046, 251938, 250353, 251831, 251835, 285886, 259419,
       251817, 232993, 252048, 250448, 266562, 252028, 250332, 232992,
       232983, 232980, 209459, 261299, 250355, 252252, 232977, 140050,
       252264, 232960, 250273, 259340, 259356, 266527, 261282, 233828,
       233789, 233176, 252645, 266713, 250244, 233783, 280234, 252876,
       261348, 261539, 250367, 251899, 250268, 261330, 251446, 234163,
       286174, 250363, 286395, 286771, 251911, 251909, 252423, 251917,
       261347, 250663, 259789, 250597, 266729, 251926, 252905, 140397,
       216067, 251833, 252854, 250270, 250659, 250607, 234202, 250636,
       250665, 251132, 252379, 215797], dtype=object)

In [ ]:
df = df_fs[top_100_features['Specs'].values]

In [ ]:
df['class'] = df_fs['class']

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [ ]:
df.to_csv('data.csv')